In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("anime-dataset-2023.csv")

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24905 entries, 0 to 24904
Data columns (total 24 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   anime_id      24905 non-null  int64 
 1   Name          24905 non-null  object
 2   English name  24905 non-null  object
 3   Other name    24905 non-null  object
 4   Score         24905 non-null  object
 5   Genres        24905 non-null  object
 6   Synopsis      24905 non-null  object
 7   Type          24905 non-null  object
 8   Episodes      24905 non-null  object
 9   Aired         24905 non-null  object
 10  Premiered     24905 non-null  object
 11  Status        24905 non-null  object
 12  Producers     24905 non-null  object
 13  Licensors     24905 non-null  object
 14  Studios       24905 non-null  object
 15  Source        24905 non-null  object
 16  Duration      24905 non-null  object
 17  Rating        24905 non-null  object
 18  Rank          24905 non-null  object
 19  Popu

In [6]:
df.columns

Index(['anime_id', 'Name', 'English name', 'Other name', 'Score', 'Genres',
       'Synopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Status',
       'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating',
       'Rank', 'Popularity', 'Favorites', 'Scored By', 'Members', 'Image URL'],
      dtype='object')

In [7]:
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
df["Scored By"] = pd.to_numeric(df["Scored By"], errors="coerce")

In [8]:
print(df["Score"].isna().sum())
print(df["Scored By"].isna().sum())

9213
9213


In [9]:
df["Score_missing"] = df["Score"].isna().astype(int)

df["Score"] = df["Score"].fillna(df["Score"].median())
df["Scored By"] = df["Scored By"].fillna(0)

In [10]:
df["Scored By"].describe()

count    2.490500e+04
mean     1.888608e+04
std      9.393985e+04
min      0.000000e+00
25%      0.000000e+00
50%      2.920000e+02
75%      3.345000e+03
max      2.660903e+06
Name: Scored By, dtype: float64

In [11]:
R = df["Score"]
v = df["Scored By"]
C = df["Score"].mean()
m = df["Scored By"].quantile(0.75)

In [12]:
df["weighted_score"] = (R*v + C*m)/(v+m)

print(df[["Name","weighted_score"]].head())

                              Name  weighted_score
0                     Cowboy Bebop        8.741375
1  Cowboy Bebop: Tengoku no Tobira        8.348149
2                           Trigun        8.202947
3               Witch Hunter Robin        7.187283
4                   Bouken Ou Beet        6.749495


In [13]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["Popularity"] = df["Popularity"].max() - df["Popularity"]

cols = ["Popularity", "Members", "Favorites"]

df[cols] = scaler.fit_transform(df[cols])

In [14]:
df["weighted_score"] = scaler.fit_transform(df[["weighted_score"]])

df["final_score"] = (
    0.85 * df["weighted_score"] +
    0.15 * df["Popularity"]
)

In [15]:
revised_df = df[["Name","Score","Scored By","weighted_score","Popularity","Members","Favorites","final_score"]].copy()

final_df = revised_df.sort_values(by = "final_score", ascending= False)

final_df.head(10)

,Name,Score,Scored By,weighted_score,Popularity,Members,Favorites,final_score
3961,Fullmetal Alchemist: Brotherhood,9.10,2020030.0,1.000000,0.999879,0.848317,1.000000,0.999982
5667,Steins;Gate,9.07,1336233.0,0.995217,0.999474,0.651714,0.840804,0.995856
14865,Shingeki no Kyojin Season 3 Part 2,9.05,1471825.0,0.992346,0.999029,0.561889,0.253876,0.993349
6456,Hunter x Hunter (2011),9.04,1651790.0,0.990963,0.999596,0.709532,0.920310,0.992257
17572,Kaguya-sama wa Kokurasetai: Ultra Romantic,9.05,451187.0,0.990332,0.991991,0.219157,0.133811,0.990580
9880,Gintama°,9.06,237957.0,0.989222,0.986612,0.159103,0.073284,0.988830
16617,Bleach: Sennen Kessen-hen,9.07,213872.0,0.990073,0.981232,0.118893,0.082714,0.988747
5989,Gintama',9.04,226175.0,0.986014,0.984387,0.140388,0.035684,0.985770
22348,Shingeki no Kyojin: The Final Season - Kankets...,9.05,155773.0,0.984925,0.980625,0.116349,0.041718,0.984280
7240,Gintama': Enchousen,9.03,157644.0,0.982114,0.971080,0.082590,0.013635,0.980459


In [16]:
print(len(df[df["Genres"] == "UNKNOWN"]))

4929


In [17]:
genre_df = df[df["Genres"].str.upper() != "UNKNOWN"].copy()

In [18]:
def get_unique_options(column):
    values = (
        df.loc[df[column].str.upper() != "UNKNOWN", column]
        .dropna()
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )
    values = values[values != ""]
    return sorted(values.unique(), key=str.lower)


unique_genres = get_unique_options("Genres")
unique_types = get_unique_options("Type")
unique_studios = get_unique_options("Studios")
unique_ratings = get_unique_options("Rating")

print("Genres:", unique_genres)
print("Total genres:", len(unique_genres))

print("\nTypes:", unique_types)
print("Total types:", len(unique_types))

print("\nStudios sample:", unique_studios[:20])
print("Total studios:", len(unique_studios))

print("\nRatings:", unique_ratings)
print("Total ratings:", len(unique_ratings))

Genres: ['Action', 'Adventure', 'Avant Garde', 'Award Winning', 'Boys Love', 'Comedy', 'Drama', 'Ecchi', 'Erotica', 'Fantasy', 'Girls Love', 'Gourmet', 'Hentai', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 'Suspense']
Total genres: 21

Types: ['Movie', 'Music', 'ONA', 'OVA', 'Special', 'TV']
Total types: 6

Studios sample: ['100studio', '10Gauge', '1IN', '2:10 AM Animation', '33 Collective', '5 Inc.', '6pucks', '7doc', '81 Produce', '8bit', 'A-1 Pictures', 'A-Line', 'A-Real', 'A.C.G.T.', 'Academy Productions', 'ACC Production', 'Acca effe', 'ACiD FiLM', 'Actas', 'Adonero']
Total studios: 1043

Ratings: ['G - All Ages', 'PG - Children', 'PG-13 - Teens 13 or older', 'R - 17+ (violence & profanity)', 'R+ - Mild Nudity', 'Rx - Hentai']
Total ratings: 6


In [19]:
from rapidfuzz import process, fuzz

def get_best_genre_match(genre, unique_genres, score_cutoff=70):
    genre = genre.lower().strip()

    # Create a mapping of lowercase genre -> original genre
    genre_map = {g.lower(): g for g in unique_genres}

    # Exact match
    if genre in genre_map:
        return genre_map[genre]

    # Fuzzy match
    match = process.extractOne(
        genre,
        genre_map.keys(),
        scorer=fuzz.WRatio,
        score_cutoff=score_cutoff
    )

    if match:
        matched_genre, score, _ = match
        print(f"Genre found. Using '{genre_map[matched_genre]}' ({score:.1f}% match)")
        return genre_map[matched_genre]

    return None

In [20]:
genre_df["Genres"] = (
    genre_df["Genres"]
    .str.lower()
    .str.split(",")
    .apply(lambda genres: [g.strip() for g in genres])
)

In [21]:
genre_df["Genre_Set"] = genre_df["Genres"].apply(set)

In [22]:
def _as_list(values):
    if values is None:
        return []
    if isinstance(values, str):
        return [values]
    return list(values)


def _split_values(value):
    if pd.isna(value):
        return []
    return [item.strip().lower() for item in str(value).split(",") if item.strip()]


def _is_known(value):
    return pd.notna(value) and str(value).strip().upper() != "UNKNOWN"


def _unique_metadata_values(column):
    return get_unique_options(column)


def _best_metadata_match(value, choices, label, score_cutoff=70):
    value = value.lower().strip()
    choice_map = {choice.lower(): choice for choice in choices}

    if value in choice_map:
        return choice_map[value]

    match = process.extractOne(
        value,
        choice_map.keys(),
        scorer=fuzz.WRatio,
        score_cutoff=score_cutoff
    )

    if match:
        matched_value, score, _ = match
        print(f"{label} found. Using '{choice_map[matched_value]}' ({score:.1f}% match)")
        return choice_map[matched_value]

    return None


def _match_inputs(values, choices, label):
    matched = []
    for value in _as_list(values):
        match = _best_metadata_match(str(value), choices, label)
        if match:
            matched.append(match.lower().strip())
    return set(matched)


def _metadata_recommend(column, values, top_n=10, label=None):
    label = label or column
    choices = _unique_metadata_values(column)
    selected = _match_inputs(values, choices, label)

    if not selected:
        print(f"No valid {label.lower()} found.")
        return None

    recommendations = df[df[column].apply(_is_known)].copy()
    recommendations[f"{column.lower()}_set"] = recommendations[column].apply(lambda value: set(_split_values(value)))
    recommendations["metadata_score"] = recommendations[f"{column.lower()}_set"].apply(
        lambda row_values: len(selected & row_values) / len(selected)
    )
    recommendations = recommendations[recommendations["metadata_score"] > 0]
    recommendations["recommendation_score"] = (
        0.7 * recommendations["metadata_score"] +
        0.3 * recommendations["final_score"]
    )

    top_recommendations = recommendations.nlargest(top_n, "recommendation_score")
    for _, row in top_recommendations.iterrows():
        print(f"{row['Name']} {row['Image URL']}")


def recommend_by_genres(genres, top_n=10):
    return _metadata_recommend("Genres", genres, top_n, label="Genre")


def recommend_by_type(type_name, top_n=10):
    return _metadata_recommend("Type", type_name, top_n, label="Type")


def recommend_by_studios(studios, top_n=10):
    return _metadata_recommend("Studios", studios, top_n, label="Studio")


def recommend_by_rating(rating, top_n=10):
    return _metadata_recommend("Rating", rating, top_n, label="Rating")


def hybrid_metadata_recommend(genres=None, type_name=None, studios=None, rating=None, top_n=10):
    filters = {
        "Genres": _match_inputs(genres, _unique_metadata_values("Genres"), "Genre"),
        "Type": _match_inputs(type_name, _unique_metadata_values("Type"), "Type"),
        "Studios": _match_inputs(studios, _unique_metadata_values("Studios"), "Studio"),
        "Rating": _match_inputs(rating, _unique_metadata_values("Rating"), "Rating"),
    }
    filters = {column: selected for column, selected in filters.items() if selected}

    if not filters:
        print("No valid metadata filters found.")
        return None

    recommendations = df.copy()
    metadata_score_columns = []

    for column, selected in filters.items():
        score_column = f"{column.lower()}_match_score"
        metadata_score_columns.append(score_column)
        recommendations[score_column] = recommendations[column].apply(
            lambda value: len(selected & set(_split_values(value))) / len(selected)
        )

    recommendations["metadata_score"] = recommendations[metadata_score_columns].mean(axis=1)
    recommendations = recommendations[recommendations["metadata_score"] > 0]
    recommendations["recommendation_score"] = (
        0.7 * recommendations["metadata_score"] +
        0.3 * recommendations["final_score"]
    )

    top_recommendations = recommendations.nlargest(top_n, "recommendation_score")
    for _, row in top_recommendations.iterrows():
        print(f"{row['Name']} {row['Image URL']}")

In [23]:
recommend_by_genres(["romance"], 20)

Kaguya-sama wa Kokurasetai: Ultra Romantic https://cdn.myanimelist.net/images/anime/1160/122627.jpg
Fruits Basket: The Final https://cdn.myanimelist.net/images/anime/1085/114792.jpg
Clannad: After Story https://cdn.myanimelist.net/images/anime/1299/110774.jpg
Monogatari Series: Second Season https://cdn.myanimelist.net/images/anime/1807/121534.jpg
Kaguya-sama wa Kokurasetai: First Kiss wa Owaranai https://cdn.myanimelist.net/images/anime/1670/130060.jpg
Howl no Ugoku Shiro https://cdn.myanimelist.net/images/anime/5/75810.jpg
Shigatsu wa Kimi no Uso https://cdn.myanimelist.net/images/anime/3/67177.jpg
Rurouni Kenshin: Meiji Kenkaku Romantan - Tsuioku-hen https://cdn.myanimelist.net/images/anime/1391/120839.jpg
Seishun Buta Yarou wa Yumemiru Shoujo no Yume wo Minai https://cdn.myanimelist.net/images/anime/1613/102179.jpg
Kimi no Suizou wo Tabetai https://cdn.myanimelist.net/images/anime/1768/93291.jpg
Fruits Basket 2nd Season https://cdn.myanimelist.net/images/anime/1972/111635.jpg
Yojou

In [24]:
recommend_by_type("TV", 20)

Fullmetal Alchemist: Brotherhood https://cdn.myanimelist.net/images/anime/1208/94745.jpg
Steins;Gate https://cdn.myanimelist.net/images/anime/1935/127974.jpg
Shingeki no Kyojin Season 3 Part 2 https://cdn.myanimelist.net/images/anime/1517/100633.jpg
Hunter x Hunter (2011) https://cdn.myanimelist.net/images/anime/1337/99013.jpg
Kaguya-sama wa Kokurasetai: Ultra Romantic https://cdn.myanimelist.net/images/anime/1160/122627.jpg
Gintama° https://cdn.myanimelist.net/images/anime/3/72078.jpg
Bleach: Sennen Kessen-hen https://cdn.myanimelist.net/images/anime/1908/135431.jpg
Gintama' https://cdn.myanimelist.net/images/anime/4/50361.jpg
Gintama': Enchousen https://cdn.myanimelist.net/images/anime/1452/123686.jpg
Fruits Basket: The Final https://cdn.myanimelist.net/images/anime/1085/114792.jpg
"Oshi no Ko" https://cdn.myanimelist.net/images/anime/1812/134736.jpg
Clannad: After Story https://cdn.myanimelist.net/images/anime/1299/110774.jpg
Gintama https://cdn.myanimelist.net/images/anime/10/73274

In [25]:
recommend_by_studios("Kyoto Animation", 50)

Koe no Katachi https://cdn.myanimelist.net/images/anime/1122/96435.jpg
Clannad: After Story https://cdn.myanimelist.net/images/anime/1299/110774.jpg
Violet Evergarden Movie https://cdn.myanimelist.net/images/anime/1825/110716.jpg
Violet Evergarden https://cdn.myanimelist.net/images/anime/1795/95088.jpg
Suzumiya Haruhi no Shoushitsu https://cdn.myanimelist.net/images/anime/1248/112352.jpg
Nichijou https://cdn.myanimelist.net/images/anime/3/75617.jpg
Violet Evergarden Gaiden: Eien to Jidou Shuki Ningyou https://cdn.myanimelist.net/images/anime/1667/112943.jpg
K-On! Movie https://cdn.myanimelist.net/images/anime/5/76233.jpg
Violet Evergarden: Kitto "Ai" wo Shiru Hi ga Kuru no Darou https://cdn.myanimelist.net/images/anime/9/89993.jpg
Kobayashi-san Chi no Maid Dragon S https://cdn.myanimelist.net/images/anime/1252/115539.jpg
Hibike! Euphonium 2 https://cdn.myanimelist.net/images/anime/10/81155.jpg
K-On!! https://cdn.myanimelist.net/images/anime/12/76121.jpg
Hyouka https://cdn.myanimelist.n